# Study 913 — Tracking-Difference Persistence 🧾

*The quant teardown. The plain-language version is [01_for_the_curious](01_for_the_curious.ipynb).*

**Two funds track the same index. One quietly returns a few basis points more
each year. Is last year's winner next year's winner?**

The fund-picker's folklore says yes: look up last year's **tracking difference** — the gap
between a fund's total return and its index's — and buy whoever won it, because good
tracking is a skill and skills persist. The duller rival rule says just buy the lowest
**expense ratio**, which is published in advance and costs nothing to read.

We test both on two families of daily **total-return** closes, as-of 2026-06-30:

- **S&P 500 ETF trio** — SPY (9.45 bp), IVV (3 bp), VOO (3 bp): 2010-09-09 →
  2026-06-30, 3,975 days, 15 complete calendar years
- **S&P 500 full ladder** — the trio plus three NAV-priced index mutual funds
  (VFIAX 4 bp, FXAIX 1.5 bp, SWPPX 2 bp), which widen the fee ladder
- **Nasdaq-100 pair** — QQQ (20 bp) and QQQM (15 bp), 5 complete years

*Numbers below are the frozen headline run (`docs/results.md`, fingerprints
`f11699949902` / `82068f45e436` / `b33e0709c240`); the only live cells run the offline
synthetic control and say so. **SPLG was requested and is unavailable** — Yahoo! Finance
serves a single stale bar for it, so it is declared missing rather than quietly swapped.*


## Design

**Estimand.** For fund *f* in complete calendar year *y*, the relative tracking difference `TD[f, y] = r[f, y] − mean_g r[g, y]`, in bp, where `r` is the compounded total return of the adjusted close. A common per-year constant shifts every fund equally, so **ranks are invariant** to the choice of index proxy (family mean, family leader, or any member); only the reported level moves — and the level is not identified from a public tape at all.

**Sample discipline.** Complete calendar years only (≥ 200 trading days), at both ends. QQQM listed 2020-10-13, so 2020 is a stub and is dropped; the as-of is 2026-06-30, so 2026 is dropped. Year-pairs are never chained across a dropped year.

**Execution.** Exactly one lag: the ranking uses returns through the last close of year *y*; the weight vector is broadcast over year *y+1*'s days and shifted forward one trading day, so it first earns on the **second** trading day of *y+1*. Costs are one-way × NAV on realised turnover. No short leg, so no borrow.

**Arms.** `winner` = argmax TD[·, y−1]; `loser` = argmin; `cheapest` = argmin published expense ratio, equal-split across ties, never traded; `leader` = the flagship, never traded; `eqw`.

> 💡 **In plain words** — rank the funds on last year's shortfall, buy the best one on the second trading day of January, pay a spread, and see whether you beat someone who bought the cheapest fund once and went to sleep.

## The proxy problem, demonstrated rather than asserted

^GSPC is a price-only index. Measured against it, every S&P 500 fund shows a 'tracking difference' of roughly +200 bp/yr — which is the dividend yield. The **level** of TD is not identified from a public tape. Note, though, that the *spread* of that column is exactly the fee ladder: the whole result, in disguise.

In [1]:
GSPC = [('SPY', 201), ('IVV', 203), ('VOO', 204), ('VFIAX', 207), ('FXAIX', 211), ('SWPPX', 205)]
print('TD vs ^GSPC (PRICE-ONLY proxy), bp/yr — this is the dividend yield:')
for name, v in GSPC:
    print('  ' + name.rjust(6) + '  ' + format(v, '+d').rjust(6) + ' bp')
vals = [v for _, v in GSPC]
print('  spread across funds: ' + str(max(vals) - min(vals))
      + ' bp  <- THIS is the fee ladder; the level is not.')


TD vs ^GSPC (PRICE-ONLY proxy), bp/yr — this is the dividend yield:
     SPY    +201 bp
     IVV    +203 bp
     VOO    +204 bp
   VFIAX    +207 bp
   FXAIX    +211 bp
   SWPPX    +205 bp
  spread across funds: 10 bp  <- THIS is the fee ladder; the level is not.


## The measurement floor

Standard deviation of relative TD against the fee spread the family actually contains, and then the same comparison against the standard error of the sample mean (sd / √years). The point is not that the ladder is invisible — it is that it is invisible **in any one year** and only emerges from averaging. A rule that reads last year's ranking is working at the left-hand column; a rule that reads the fee sheet skips the estimation problem entirely.

The two ~40 bp 'outlier' years in the trio (VOO 2014, IVV 2016) fully reverse the next year and are dividend-timing artefacts of the adjusted close, not slippage.

In [2]:
FLOOR = [('S&P 500 ETF trio', 10.7, 6.45, 15), ('S&P 500 full ladder', 9.7, 7.95, 14), ('Nasdaq-100 pair', 5.0, 5.0, 5)]
print('family'.rjust(22) + 'sd(TD)'.rjust(10) + 'fee spread'.rjust(12)
      + 'one year?'.rjust(11) + 'se of mean'.rjust(12) + 'full sample?'.rjust(13))
for name, sd, spread, years in FLOOR:
    se = sd / years ** 0.5
    print(name.rjust(22) + (format(sd, '.1f') + ' bp').rjust(10)
          + (format(spread, '.2f') + ' bp').rjust(12)
          + ('yes' if spread > sd else 'NO').rjust(11)
          + (format(se, '.2f') + ' bp').rjust(12)
          + ('yes' if spread > se else 'NO').rjust(13))


                family    sd(TD)  fee spread  one year?  se of mean full sample?
      S&P 500 ETF trio   10.7 bp     6.45 bp         NO     2.76 bp          yes
   S&P 500 full ladder    9.7 bp     7.95 bp         NO     2.59 bp          yes
       Nasdaq-100 pair    5.0 bp     5.00 bp         NO     2.24 bp          yes


## Persistence: rank correlation, a *t*, and a permutation null

The *t* is a one-sample *t* on the mean of the per-pair Spearman coefficients (n = year-pairs; there is no daily autocorrelation to correct here, so no HAC). The permutation null shuffles the **order of the calendar years**, preserving each year's cross-section and destroying only the time linkage.

> 💡 **In plain words** — the permutation asks whether the link is specifically between *consecutive* years, or whether any year predicts any other equally well. If the latter, it is a constant — and a constant you can look up is not a forecast.

In [3]:
P = [('S&P 500 ETF trio', 0.071, 0.43, 14, 0.964, 0.305, -0.179), ('S&P 500 full ladder', 0.253, 2.34, 13, 0.307, 0.195, 0.2)]
print('family'.rjust(22) + 'rho(y,y+1)'.rjust(12) + 't'.rjust(8)
      + 'pairs'.rjust(7) + 'perm p'.rjust(9) + 'rho(all pairs)'.rjust(16)
      + 'residual rho'.rjust(14))
for name, rho, t, pairs, pp, allp, res in P:
    print(name.rjust(22) + format(rho, '+.3f').rjust(12) + format(t, '+.2f').rjust(8)
          + str(pairs).rjust(7) + format(pp, '.3f').rjust(9)
          + format(allp, '+.3f').rjust(16) + format(res, '+.3f').rjust(14))
print()
print('Trio  : no persistence at all (permutation p = 0.96).')
print('Ladder: rho = +0.253 (t = +2.34) BUT permutation p = 0.31, and the all-pairs')
print('        rank correlation (+0.195) matches the consecutive-pair one (+0.253).')
print('     => a time-invariant LEVEL (the fee sheet), not a year-to-year memory.')


                family  rho(y,y+1)       t  pairs   perm p  rho(all pairs)  residual rho
      S&P 500 ETF trio      +0.071   +0.43     14    0.964          +0.305        -0.179
   S&P 500 full ladder      +0.253   +2.34     13    0.307          +0.195        +0.200

Trio  : no persistence at all (permutation p = 0.96).
Ladder: rho = +0.253 (t = +2.34) BUT permutation p = 0.31, and the all-pairs
        rank correlation (+0.195) matches the consecutive-pair one (+0.253).
     => a time-invariant LEVEL (the fee sheet), not a year-to-year memory.


Two caveats on that table. Fund-demeaning over 13–14 observations imposes a known negative small-sample bias on the residual autocorrelation, so a residual near zero is the expected reading under 'fees and nothing else'; the ladder's +0.200 (*t* = +1.71) does not clear the bar. And with two funds (QQQ/QQQM) Spearman degenerates to ±1, so the Nasdaq result is a **5/5 sign run** and is reported as one, not as a rank test.

## The rules, in two units

`t (annual)` is the paired *t* on the per-year gap; `t (ann HAC)` is Newey-West with one annual lag on the same series. Calendar years do not overlap, so the two agree — the HAC column is there so independence is never simply assumed.

In [4]:
G = [('trio', 'winner-cheapest', 0.33, 0.1, 0.1, 0.03, '7/14'), ('trio', 'cheapest-leader', 3.27, 1.26, 1.4, 0.52, '12/14'), ('ladder', 'winner-cheapest', -6.08, -1.03, -1.22, -0.89, '3/13'), ('ladder', 'cheapest-leader', 10.64, 4.85, 4.78, 1.32, '11/13'), ('ndx', 'cheapest-leader', 8.79, 2.93, 4.32, 0.28, '4/4')]
print('family'.rjust(8) + 'gap'.rjust(18) + 'bp/yr'.rjust(10)
      + 't (annual)'.rjust(12) + 't (ann HAC)'.rjust(13)
      + 't (daily HAC)'.rjust(15) + 'yrs +'.rjust(8))
for fam, gap, bp, t_ann, t_hac, t_day, pos in G:
    print(fam.rjust(8) + gap.rjust(18) + format(bp, '+.2f').rjust(10)
          + format(t_ann, '+.2f').rjust(12) + format(t_hac, '+.2f').rjust(13)
          + format(t_day, '+.2f').rjust(15) + pos.rjust(8))
print()
print('The rule window is one year shorter than the measurement window: the first')
print('complete year is consumed by the ranking. The Nasdaq 5/5 raw-TD run is a 4/4')
print('once it has to be traded, and its t = +2.93 is computed on those four years.')


  family               gap     bp/yr  t (annual)  t (ann HAC)  t (daily HAC)   yrs +
    trio   winner-cheapest     +0.33       +0.10        +0.10          +0.03    7/14
    trio   cheapest-leader     +3.27       +1.26        +1.40          +0.52   12/14
  ladder   winner-cheapest     -6.08       -1.03        -1.22          -0.89    3/13
  ladder   cheapest-leader    +10.64       +4.85        +4.78          +1.32   11/13
     ndx   cheapest-leader     +8.79       +2.93        +4.32          +0.28     4/4

The rule window is one year shorter than the measurement window: the first
complete year is consumed by the ranking. The Nasdaq 5/5 raw-TD run is a 4/4
once it has to be traded, and its t = +2.93 is computed on those four years.


**Why the two units disagree, and which to believe.** The daily HAC *t* is near zero everywhere — *including* for gaps that are strongly significant annually. That is a power problem, not a contradiction: the day-to-day difference between two funds holding the same basket is dominated by print timing and dividend ex-date offsets, noise that reverses within days and swamps a 10 bp/yr drift. The estimand is defined once a year, so the annual paired test is its natural unit; the daily HAC figure is reported as the conservative floor rather than suppressed.

Block-bootstrap 95% CIs on the annual gap (5,000 draws, 2-year blocks): ladder cheapest−leader **[+6.1, +14.8]** and Nasdaq **[+4.5, +13.1]** are clear of zero; ladder winner−cheapest **[-17.4, +0.9]** and trio winner−cheapest **[-5.9, +6.6]** are not.

## The one look-ahead in the study, priced

Nothing in the *return* series peeks: the ranking formed at the last close of year *y* first earns on the second trading day of *y+1*. But the fund the `cheapest` arm holds is picked from the fee sheet published **today**, and FXAIX has charged 1.5 bp only since 2019 (SWPPX 2 bp since 2017, IVV 3 bp since 2016). The direction was public in advance — SPY and QQQ have always been the dearest of their families — but *which* low-fee rung ends up lowest is chosen ex post. So the +10.64 bp/yr headline is, in substance, **FXAIX selected with hindsight**.

The control removes the selection entirely: race every non-flagship fund against the flagship, buy-and-hold, plus the equal-weight blend of all of them — a rule needing no fee sheet, no ranking and no forecast (`strategy.per_fund_gap_vs_leader`).

In [5]:
H = [('IVV - SPY', 2.37, 0.62, '12/14'), ('VOO - SPY', 3.46, 0.75, '12/14'), ('SWPPX - SPY', 4.47, 1.59, '10/14'), ('VFIAX - SPY', 6.04, 2.49, '10/14'), ('FXAIX - SPY', 9.71, 4.38, '11/14'), ('EQW of all five - SPY', 5.21, 2.68, '11/14'), ('QQQM - QQQ', 8.55, 3.95, '5/5')]
print('Buy-and-hold vs the family flagship — NO fee sheet, no ranking, no forecast')
print('gap'.rjust(24) + 'bp/yr'.rjust(10) + 't (annual)'.rjust(12)
      + 'yrs +'.rjust(8))
for name, bp, t, pos in H:
    print(name.rjust(24) + format(bp, '+.2f').rjust(10)
          + format(t, '+.2f').rjust(12) + pos.rjust(8))
print()
print('1. EVERY fund beats its flagship: the SIGN owes nothing to hindsight.')
print('2. The hindsight-free magnitude is +5.21 bp/yr (t = +2.68) — about HALF the')
print('   +10.64 the fee-sheet rule reports. Half the headline is the fund pick.')
print('3. The two ETFs already cheap in 2012 (IVV, VOO) do not clear t = 1.')
print('The Nasdaq pair has no selection to make, and is the cleaner reading.')


Buy-and-hold vs the family flagship — NO fee sheet, no ranking, no forecast
                     gap     bp/yr  t (annual)   yrs +
               IVV - SPY     +2.37       +0.62   12/14
               VOO - SPY     +3.46       +0.75   12/14
             SWPPX - SPY     +4.47       +1.59   10/14
             VFIAX - SPY     +6.04       +2.49   10/14
             FXAIX - SPY     +9.71       +4.38   11/14
   EQW of all five - SPY     +5.21       +2.68   11/14
              QQQM - QQQ     +8.55       +3.95     5/5

1. EVERY fund beats its flagship: the SIGN owes nothing to hindsight.
2. The hindsight-free magnitude is +5.21 bp/yr (t = +2.68) — about HALF the
   +10.64 the fee-sheet rule reports. Half the headline is the fund pick.
3. The two ETFs already cheap in 2012 (IVV, VOO) do not clear t = 1.
The Nasdaq pair has no selection to make, and is the cleaner reading.


## Sharpe is the wrong instrument here

In [6]:
S = [('winner', 0.823), ('cheapest', 0.8228), ('leader', 0.8241), ('eqw', 0.8234)]
print('Excess-of-cash Sharpe vs BIL (both arms excess), S&P 500 ETF trio:')
for name, v in S:
    print('  ' + name.rjust(9) + '  ' + format(v, '.4f'))
vals = [v for _, v in S]
print('  spread: ' + format(max(vals) - min(vals), '.4f'))
print()
print('Every arm holds the same index, so they share ~99.99% of their variance.')
print('A genuine 3-11 bp/yr edge lives in the 4th decimal of the Sharpe ratio and is')
print('unmeasurable there. A tracking-difference edge is a return GAP, not a Sharpe.')


Excess-of-cash Sharpe vs BIL (both arms excess), S&P 500 ETF trio:
     winner  0.8230
   cheapest  0.8228
     leader  0.8241
        eqw  0.8234
  spread: 0.0013

Every arm holds the same index, so they share ~99.99% of their variance.
A genuine 3-11 bp/yr edge lives in the 4th decimal of the Sharpe ratio and is
unmeasurable there. A tracking-difference edge is a return GAP, not a Sharpe.


## Era cut

In [7]:
E = [('trio   cheapest-leader', '2012-2018', -0.55, -0.12, '2019-2025', 7.09, 4.02), ('ladder cheapest-leader', '2013-2018', 10.94, 3.52, '2019-2025', 10.39, 3.14)]
for name, e_lab, e_gap, e_t, l_lab, l_gap, l_t in E:
    print(name + ':  ' + e_lab + ' ' + format(e_gap, '+.2f') + ' bp/yr (t='
          + format(e_t, '+.2f') + ')   ' + l_lab + ' ' + format(l_gap, '+.2f')
          + ' bp/yr (t=' + format(l_t, '+.2f') + ')')
print()
print('The ladder gap is era-robust (+10.94 then +10.39).')
print('The trio gap only appears late: SPY has not cut its fee while its rivals have.')


trio   cheapest-leader:  2012-2018 -0.55 bp/yr (t=-0.12)   2019-2025 +7.09 bp/yr (t=+4.02)
ladder cheapest-leader:  2013-2018 +10.94 bp/yr (t=+3.52)   2019-2025 +10.39 bp/yr (t=+3.14)

The ladder gap is era-robust (+10.94 then +10.39).
The trio gap only appears late: SPY has not cut its fee while its rivals have.


## Cost sweep and the tax assumption

`cheapest` never switches fund, so its gap is cost-invariant by construction; `winner` changed fund 10 times in 14 live years, so it pays. One disclosure: where `cheapest` is a **tie** (the trio's IVV/VOO at 3 bp) it is held at a constant 50/50 target — an implicit daily rebalance that is charged nothing. Between funds this similar the freebie measures **+0.06 bp/yr**, so it changes no conclusion, but only the single-fund arms are literally buy-and-hold.

In [8]:
SW = [(0.0, 2.01, 0.6), (1.0, 0.33, 0.1), (5.0, -6.39, -1.77), (10.0, -14.79, -3.35), (25.0, -39.98, -4.92)]
print('winner - cheapest, S&P 500 ETF trio, by one-way switching cost')
print('cost (bp)'.rjust(10) + 'gap bp/yr'.rjust(12) + 't (annual)'.rjust(12))
for c, gap, t in SW:
    print(format(c, '.0f').rjust(10) + format(gap, '+.2f').rjust(12)
          + format(t, '+.2f').rjust(12))
print()
print('A coin flip gross; reliably negative at any cost a real switch incurs.')


winner - cheapest, S&P 500 ETF trio, by one-way switching cost
 cost (bp)   gap bp/yr  t (annual)
         0       +2.01       +0.60
         1       +0.33       +0.10
         5       -6.39       -1.77
        10      -14.79       -3.35
        25      -39.98       -4.92

A coin flip gross; reliably negative at any cost a real switch incurs.


In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from td_persist import strategy as st
gap = 9.45 - 3.00   # published SPY minus VOO expense ratio — an ASSUMPTION
print('Years for the ' + format(gap, '.2f')
      + ' bp/yr fee gap to repay a realised-gain tax bill')
print('(ASSUMPTION grid: neither the embedded gain nor the rate is on the tape)')
print(st.tax_breakeven_years(gap).round(0).to_string())


Years for the 6.45 bp/yr fee gap to repay a realised-gain tax bill
(ASSUMPTION grid: neither the embedded gain nor the rate is on the tape)
                   tax_000  tax_150  tax_200  tax_238
embedded_gain_pct                                    
10.0                   0.0     23.0     31.0     37.0
25.0                   0.0     58.0     78.0     92.0
50.0                   0.0    116.0    155.0    184.0
100.0                  0.0    233.0    310.0    369.0


## Synthetic control — the machinery, and only the machinery

**Synthetic data, not the real tape.** A panel of four funds tracking one index, separated by a planted fee ladder against an 8 bp annual noise floor. At `signal_strength=1` the ladder is 30 bp and the pipeline must find it; at `signal_strength=0` every fund charges the identical fee and the pipeline must stay silent. Eight seeds each — never one lucky draw.

In [10]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import pandas as pd
from td_persist import data, strategy as st
out = []
for ss in (1.0, 0.0):
    rows = [st.synthetic_detect(*data.synthetic_panel(signal_strength=ss, seed=913 + s))
            for s in range(8)]
    f = pd.DataFrame(rows)
    out.append({'world': 'planted 30 bp ladder' if ss else 'flat-fee null',
                'rho': f['mean_spearman'].mean(),
                'rho_min': f['mean_spearman'].min(),
                'rho_max': f['mean_spearman'].max(),
                'cheapest-leader bp': f['cheapest_minus_leader_bp'].mean(),
                't': f['t_cheapest_minus_leader'].mean()})
print('SYNTHETIC control (8 seeds each) — never supports a real-tape stamp')
print(pd.DataFrame(out).set_index('world').round(3).to_string())


SYNTHETIC control (8 seeds each) — never supports a real-tape stamp
                        rho  rho_min  rho_max  cheapest-leader bp       t
world                                                                    
planted 30 bp ladder  0.725    0.415    0.877              35.661  10.528
flat-fee null        -0.079   -0.369    0.138              -0.291  -0.160


## Threats to validity, named

- **Survivorship.** Eight vehicles, all of which still exist today. Trackers that tracked badly enough to close or merge are absent by construction, so measured TD dispersion is understated and the cheapest-fund result sits on the friendliest possible sample. Named on the Signal axis.
- **Expense ratios are an ASSUMPTION that carries hindsight**, taken from issuer disclosure at build time; several were cut inside the sample, so today's ladder is a level rather than a history, and the `cheapest` arm's fund choice is partly ex post. The cost sweep bounds how wrong the assumed spread can be, and the hindsight-free control above prices the selection: the gap halves (+10.64 → +5.21 bp/yr) but keeps its sign and its *t* (+2.68).
- **Multi-fund arms are held at target weight**, i.e. implicitly rebalanced daily at no cost. On the real trio's tied `cheapest` (IVV/VOO) that freebie is +0.06 bp/yr — immaterial here, but only single-fund arms are literally buy-and-hold.
- **NAV-priced legs.** VFIAX, FXAIX and SWPPX carry no spread and no premium/discount, which is exactly why they widen the ladder cleanly — but they cannot be switched intraday, and the tax grid applies to them identically.
- **SPLG is missing, not omitted.** The source serves a single stale bar for it; it is declared in `data.UNAVAILABLE` rather than silently substituted.
- **Five complete years** is all QQQ/QQQM offers. A 4/4 sign run is suggestive and fully consistent with the 5 bp fee gap — it is not a large-sample result.

## Verdict

- **Signal — Mixed.** The persistence claim fails where it matters: the S&P 500 ETF trio shows rank persistence of **+0.071** (*t* = +0.43, permutation *p* = 0.96), because a 6.45 bp fee spread is unresolvable through a 10.7 bp measurement floor. Across a wider ladder the *fee* effect is unambiguous — cheapest − leader **+10.64 bp/yr, *t* = +4.85** (HAC +4.78), 11/13 years, era-robust (+10.94 then +10.39), CI [+6.1, +14.8], halving to **+5.21 (*t* = +2.68)** once the ex-post fee-sheet pick is removed — and QQQM beat QQQ by **+8.55 bp/yr (*t* = +3.95)** in 5/5 years, +8.79 (*t* = +2.93) over the 4/4 a rule could trade. But the permutation test identifies that as a time-invariant level rather than a memory: the all-pairs rank correlation (+0.195) matches the consecutive-pair one (+0.253). Half the claim is right, for a reason that makes the other half redundant. Survivorship flatters the sample, and half the fee-gap magnitude is fee-sheet hindsight; the synthetic control fires on a planted ladder (+0.725, gap +35.7 bp) and stays silent on the null (-0.079, gap -0.29 bp).
- **Tradability — Fragile.** The bankable form is a purchase decision, not a strategy: buy a cheap share class, never trade it, collect 3.3–8.6 bp/yr. The rotation form pays +0.33 bp/yr (*t* = +0.10) for 10 trades in 14 years, is negative beyond 1 bp of switching cost, and — at a +50% embedded gain and a 20% rate — takes **155 years** to repay the tax on moving an existing position.